Use Case 2 tasks

Data Cleaning and Preprocessing

Import libraries and set paths

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_DIR = Path.cwd().parent

RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

Load the repaired CSV dataset we validated in Use Case 1.

In [2]:
repaired_file = PROCESSED_DATA_DIR / "chicago_crime_dataset_repaired.csv"

crime_df = pd.read_csv(
    repaired_file,
    low_memory=False
)

print("Crime dataset loaded successfully.")
print("Rows:", crime_df.shape[0])
print("Columns:", crime_df.shape[1])

Crime dataset loaded successfully.
Rows: 498
Columns: 20


In [33]:
cleaned_df = crime_df.copy()

print("Working copy created.")
print("Original rows:", len(crime_df))
print("Working copy rows:", len(cleaned_df))

display(cleaned_df.head(10))

Working copy created.
Original rows: 498
Working copy rows: 498


,id,case_number,date,block,iucr_code,location_desc,arrest,domestic,beat_num,district_code,ward_no,community_code,fbi_code,x_coordinate,y_coordinate,year,date_of_update,latitude,longitude,location
0,10508693,HZ250496,5/3/2016,013XX S SAWYER AVE,486,APARTMENT,True,True,1022,10,24,29,08B,1154907.0,1893681.0,2016,5/10/2016,41.864073,-87.706819,"(41.864073157, -87.706818608)"
1,10508695,HZ250409,5/3/2016,061XX S DREXEL AVE,486,RESIDENCE,False,True,313,3,20,42,08B,1183066.0,1864330.0,2016,5/10/2016,41.782922,-87.604363,"(41.782921527, -87.60436317)"
2,10508697,HZ250503,5/3/2016,053XX W CHICAGO AVE,470,STREET,False,False,1524,15,37,25,24,1140789.0,1904819.0,2016,5/10/2016,41.894908,-87.758372,"(41.894908283, -87.758371958)"
3,10508698,HZ250424,5/3/2016,049XX W FULTON ST,460,SIDEWALK,False,False,1532,15,28,25,08B,1143223.0,1901475.0,2016,5/10/2016,41.885687,-87.749516,"(41.885686845, -87.749515983)"
4,10508699,HZ250455,5/3/2016,003XX N LOTUS AVE,820,RESIDENCE,False,True,1523,15,28,25,6,1139890.0,1901675.0,2016,5/10/2016,41.886297,-87.761751,"(41.886297242, -87.761750709)"
5,10508702,HZ250447,5/3/2016,082XX S MARYLAND AVE,041A,STREET,False,False,631,6,8,44,04B,1183336.0,1850642.0,2016,5/10/2016,41.745354,-87.603799,"(41.745354023, -87.603798903)"
6,10508703,HZ250489,5/3/2016,027XX S STATE ST,460,CHA HALLWAY/STAIRWELL/ELEVATOR,False,False,133,1,3,35,08B,1176730.0,1886544.0,2016,5/10/2016,41.844024,-87.626923,"(41.844023772, -87.626923253)"
7,10508704,HZ250514,5/3/2016,002XX E 46TH ST,460,RESIDENCE PORCH/HALLWAY,False,False,215,2,3,38,08B,1178514.0,1874573.0,2016,5/10/2016,41.811134,-87.620741,"(41.811133958, -87.62074077)"
8,10508709,HZ250523,5/3/2016,014XX W DEVON AVE,460,SIDEWALK,False,False,2432,24,40,1,08B,1165696.0,1942616.0,2016,5/10/2016,41.998131,-87.665814,"(41.99813061, -87.665814038)"
9,10508982,HZ250667,5/3/2016,069XX S ASHLAND AVE,486,STREET,False,True,735,7,17,67,08B,1166876.0,1858796.0,2016,5/10/2016,41.768097,-87.663879,"(41.768096835, -87.663878589)"


Inspect column names and data types

In [4]:
print("Column names:")
if "cleaned_df" not in globals():
    repaired_file = RAW_DATA_DIR.parent / "processed" / "chicago_crime_dataset_repaired.csv"
    cleaned_df = pd.read_csv(repaired_file, low_memory=False).copy()

print(cleaned_df.columns.tolist())

print("\nData types:")
display(cleaned_df.dtypes.to_frame(name="Data Type"))

Column names:
['id', 'case_number', 'date', 'block', 'iucr_code', 'location_desc', 'arrest', 'domestic', 'beat_num', 'district_code', 'ward_no', 'community_code', 'fbi_code', 'x_coordinate', 'y_coordinate', 'year', 'date_of_update', 'latitude', 'longitude', 'location']

Data types:


,Data Type
id,int64
case_number,str
date,str
block,str
iucr_code,str
location_desc,str
arrest,bool
domestic,bool
beat_num,int64
district_code,int64


Remove accidental spaces from column names

In [5]:
cleaned_df.columns = cleaned_df.columns.str.strip()

print("Updated column names:")
print(cleaned_df.columns.tolist())

Updated column names:
['id', 'case_number', 'date', 'block', 'iucr_code', 'location_desc', 'arrest', 'domestic', 'beat_num', 'district_code', 'ward_no', 'community_code', 'fbi_code', 'x_coordinate', 'y_coordinate', 'year', 'date_of_update', 'latitude', 'longitude', 'location']


Check missing values

In [6]:
missing_summary = pd.DataFrame({
    "Missing Count": cleaned_df.isnull().sum(),
    "Missing Percentage": (
        cleaned_df.isnull().mean() * 100
    ).round(2)
})

display(
    missing_summary.sort_values(
        by="Missing Percentage",
        ascending=False
    )
)

,Missing Count,Missing Percentage
location,29,5.82
y_coordinate,29,5.82
longitude,29,5.82
latitude,29,5.82
x_coordinate,29,5.82
location_desc,4,0.80
case_number,0,0.00
id,0,0.00
block,0,0.00
date,0,0.00


Identify columns with more than 50% missing values

In [7]:
high_missing_columns = (
    missing_summary[
        missing_summary["Missing Percentage"] > 50
    ]
)

print("Columns with more than 50% missing values:")
display(high_missing_columns)

Columns with more than 50% missing values:


,Missing Count,Missing Percentage


Inspect missing values in important columns

In [8]:
important_columns = [
    "case_number",
    "date",
    "primary_type",
    "iucr_code",
    "arrest",
    "community_code",
    "district_code",
    "beat_num"
]

available_important_columns = [
    column for column in important_columns
    if column in cleaned_df.columns
]

display(
    cleaned_df[available_important_columns]
    .isnull()
    .sum()
    .to_frame(name="Missing Count")
)

,Missing Count
case_number,0
date,0
iucr_code,0
arrest,0
community_code,0
district_code,0
beat_num,0


Check missing values in key fields

In [9]:
key_columns = [
    "case_number",
    "date",
    "iucr_code"
]

for column in key_columns:
    if column in cleaned_df.columns:
        print(
            column,
            "missing:",
            cleaned_df[column].isnull().sum()
        )

case_number missing: 0
date missing: 0
iucr_code missing: 0


Remove records with missing essential identifiers

We will treat case_number and date as essential fields for this initial cleaning step

In [10]:
essential_columns = ["case_number", "date"]

missing_essential = cleaned_df[
    essential_columns
].isnull().any(axis=1)

print("Records missing case number or date:", missing_essential.sum())
print("Records before removal:", len(cleaned_df))

cleaned_df = cleaned_df.loc[~missing_essential].copy()

print("Records after removal:", len(cleaned_df))

Records missing case number or date: 0
Records before removal: 498
Records after removal: 498


Convert the date column into datetime format

In [11]:
cleaned_df["date"] = pd.to_datetime(
    cleaned_df["date"],
    errors="coerce"
)

print("Date conversion completed.")

print("Invalid or missing dates:", cleaned_df["date"].isnull().sum())

display(cleaned_df[["case_number", "date"]].head(10))

Date conversion completed.
Invalid or missing dates: 0


,case_number,date
0,HZ250496,2016-05-03
1,HZ250409,2016-05-03
2,HZ250503,2016-05-03
3,HZ250424,2016-05-03
4,HZ250455,2016-05-03
5,HZ250447,2016-05-03
6,HZ250489,2016-05-03
7,HZ250514,2016-05-03
8,HZ250523,2016-05-03
9,HZ250667,2016-05-03


Identify invalid dates and Remove records with invalid dates

In [12]:
invalid_date_rows = cleaned_df[
    cleaned_df["date"].isnull()
]

print("Number of invalid dates:", len(invalid_date_rows))

display(invalid_date_rows.head(10))


before_date_cleaning = len(cleaned_df)

cleaned_df = cleaned_df.dropna(
    subset=["date"]
).copy()

after_date_cleaning = len(cleaned_df)

print("Records removed:", before_date_cleaning - after_date_cleaning)
print("Records remaining:", after_date_cleaning)

Number of invalid dates: 0


,id,case_number,date,block,iucr_code,location_desc,arrest,domestic,beat_num,district_code,ward_no,community_code,fbi_code,x_coordinate,y_coordinate,year,date_of_update,latitude,longitude,location


Records removed: 0
Records remaining: 498


Standardize categorical text columns by removing unnecessary leading and trailing spaces

In [13]:
categorical_columns = [
    "block",
    "location_desc",
    "fbi_code"
]

for column in categorical_columns:
    if column in cleaned_df.columns:
        cleaned_df[column] = (
            cleaned_df[column]
            .astype("string")
            .str.strip()
            .str.upper()
        )

print("Categorical text standardization completed.")

Categorical text standardization completed.


Standardize Boolean columns

In [14]:
boolean_columns = ["arrest", "domestic"]

for column in boolean_columns:
    if column in cleaned_df.columns:
        cleaned_df[column] = (
            cleaned_df[column]
            .astype("string")
            .str.strip()
            .str.upper()
            .map({
                "TRUE": True,
                "FALSE": False
            })
        )

print("Boolean standardization completed.")

for column in boolean_columns:
    if column in cleaned_df.columns:
        print(f"\n{column}:")
        print(cleaned_df[column].value_counts(dropna=False))

Boolean standardization completed.

arrest:
arrest
False    374
True     124
Name: count, dtype: int64[pyarrow]

domestic:
domestic
False    430
True      68
Name: count, dtype: int64[pyarrow]


Create Year, Month and Day of Week columns

In [15]:
cleaned_df["Year"] = cleaned_df["date"].dt.year
cleaned_df["Month"] = cleaned_df["date"].dt.month
cleaned_df["DayOfWeek"] = cleaned_df["date"].dt.day_name()

display(
    cleaned_df[
        ["date", "Year", "Month", "DayOfWeek"]
    ].head(10)
)

,date,Year,Month,DayOfWeek
0,2016-05-03,2016,5,Tuesday
1,2016-05-03,2016,5,Tuesday
2,2016-05-03,2016,5,Tuesday
3,2016-05-03,2016,5,Tuesday
4,2016-05-03,2016,5,Tuesday
5,2016-05-03,2016,5,Tuesday
6,2016-05-03,2016,5,Tuesday
7,2016-05-03,2016,5,Tuesday
8,2016-05-03,2016,5,Tuesday
9,2016-05-03,2016,5,Tuesday


Calculate percentages of missing values per column using NumPy.

This directly addresses the NumPy missing-value calculation requirement.

In [16]:
missing_counts = np.isnan(
    cleaned_df.select_dtypes(include=["number"]).to_numpy()
).sum(axis=0)

numeric_columns = cleaned_df.select_dtypes(
    include=["number"]
).columns

numeric_missing_percentage = (
    missing_counts / len(cleaned_df)
) * 100

numeric_missing_summary = pd.DataFrame({
    "Column": numeric_columns,
    "Missing Count": missing_counts,
    "Missing Percentage": numeric_missing_percentage.round(2)
})

display(
    numeric_missing_summary.sort_values(
        by="Missing Percentage",
        ascending=False
    )
)

,Column,Missing Count,Missing Percentage
8,latitude,29,5.82
9,longitude,29,5.82
6,y_coordinate,29,5.82
5,x_coordinate,29,5.82
3,ward_no,0,0.00
2,district_code,0,0.00
1,beat_num,0,0.00
0,id,0,0.00
7,year,0,0.00
4,community_code,0,0.00


Check duplicate records

In [17]:
print("Duplicate rows:", cleaned_df.duplicated().sum())

print(
    "Duplicate case numbers:",
    cleaned_df["case_number"].duplicated().sum()
)

print("Cleaned dataset dimensions:", cleaned_df.shape)


Duplicate rows: 0
Duplicate case numbers: 0
Cleaned dataset dimensions: (498, 23)


Check unique crime-related codes exist in the dataset

In [18]:
print("Unique IUCR codes:", cleaned_df["iucr_code"].nunique())

print("\nMost common IUCR codes:")
display(
    cleaned_df["iucr_code"]
    .value_counts()
    .head(10)
)

Unique IUCR codes: 97

Most common IUCR codes:


iucr_code
486     43
820     42
460     27
1310    24
1320    23
810     22
890     21
560     21
910     16
860     16
Name: count, dtype: int64

Check date range and potential date format anomalies

In [19]:
print("Earliest date:", cleaned_df["date"].min())
print("Latest date:", cleaned_df["date"].max())

print("\nRecords by year:")
display(cleaned_df["Year"].value_counts().sort_index())

print("\nRecords outside 2012–2016:")
display(
    cleaned_df[
        ~cleaned_df["Year"].between(2012, 2016)
    ][["case_number", "date", "Year"]]
)

Earliest date: 2012-01-01 00:00:00
Latest date: 2016-05-04 00:00:00

Records by year:


Year
2012      2
2013      1
2014      1
2015      4
2016    490
Name: count, dtype: int64


Records outside 2012–2016:


,case_number,date,Year


Create a data-quality report

This records the missing-coordinate issue for project documentation.

In [20]:
coordinate_columns = [
    "latitude",
    "longitude",
    "x_coordinate",
    "y_coordinate"
]

display(cleaned_df[coordinate_columns].isnull().sum())

# Count records missing all four coordinate fields
all_coordinates_missing = cleaned_df[
    coordinate_columns
].isnull().all(axis=1)

# Count records missing at least one coordinate field
any_coordinate_missing = cleaned_df[
    coordinate_columns
].isnull().any(axis=1)

print(
    "Records missing all four coordinates:",
    all_coordinates_missing.sum()
)

print(
    "Records missing at least one coordinate:",
    any_coordinate_missing.sum()
)

coordinate_quality_report = pd.DataFrame({
    "Column": coordinate_columns,
    "Missing Count": [
        cleaned_df[column].isnull().sum()
        for column in coordinate_columns
    ],
    "Missing Percentage": [
        round(cleaned_df[column].isnull().mean() * 100, 2)
        for column in coordinate_columns
    ],
    "Treatment": [
        "Retained as missing; no imputation"
        for column in coordinate_columns
    ]
})

display(coordinate_quality_report)

latitude        29
longitude       29
x_coordinate    29
y_coordinate    29
dtype: int64

Records missing all four coordinates: 29
Records missing at least one coordinate: 29


,Column,Missing Count,Missing Percentage,Treatment
0,latitude,29,5.82,Retained as missing; no imputation
1,longitude,29,5.82,Retained as missing; no imputation
2,x_coordinate,29,5.82,Retained as missing; no imputation
3,y_coordinate,29,5.82,Retained as missing; no imputation


Save the final cleaned dataset

In [21]:
cleaned_file = PROCESSED_DATA_DIR / "chicago_crime_dataset_cleaned.csv"

cleaned_df.to_csv(
    cleaned_file,
    index=False
)

print("Final cleaned dataset saved:", cleaned_file)
print("Total records:", len(cleaned_df))
print("Total columns:", len(cleaned_df.columns))

Final cleaned dataset saved: c:\Users\sapni\OneDrive\Desktop\Chicago_Crime_Capstone\data\processed\chicago_crime_dataset_cleaned.csv
Total records: 498
Total columns: 23
